# Pipeline Integrado de Obtención, Inyección y Limpieza de Datos

**Proyecto:** Smart Kitchen Intelligence (SKI)  
**Objetivo:** Ejecutar el pipeline completo de preparación de datos

## Flujo del Pipeline

```
1. extract_patterns.py
   ↓
2. simulation.py → movements_raw.csv
   ↓
3. ingestion.py (NUEVA: OpenFoodFacts) → catalog_raw.csv
   ↓
4. anomaly_injection.py → movements_with_anomalies.csv
   ↓
5. preprocessing.py (MEJORADO) → inventory_v1.csv (LIMPIO)
   ↓
6. Análisis exploratorio → Tableau
```

## Cambios Realizados

### ✅ Fase 1: Actualización de Ingestion
- **Antes:** USDA API (legacy)
- **Después:** OpenFoodFacts API (actual)
- **Beneficio:** Mejor cobertura nutricional y Nutriscore nativo

### ✅ Fase 2: Inyección de Anomalías
- Nuevo módulo: `src/anomaly_injection.py`
- Simula datos reales "sucios"
- Anomalías inyectadas:
  - 15% valores nulos
  - 5% duplicados
  - 8% outliers
  - 3% inconsistencias lógicas
  - 5% errores de tipo

### ✅ Fase 3: Limpieza Robusta
- Mejorado: `src/preprocessing.py`
- Detección automática de anomalías
- Bitácora detallada de transformaciones
- Validación QA antes/después


## 1. CONFIGURACIÓN INICIAL

In [ ]:
import sys
import os
from pathlib import Path

# Este notebook vive en notebooks/, por lo que el cwd del kernel de Jupyter es
# notebooks/ (no la raiz del repo). Los scripts en src/ usan rutas relativas
# del tipo 'data/raw/...' asumiendo que se ejecutan DESDE la raiz del repo, asi
# que fijamos REPO_ROOT explicitamente y lo usamos tanto para las rutas que
# lee/escribe este notebook como para el cwd de los subprocess.run() de abajo.
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

# Crear directorios necesarios (rutas absolutas via REPO_ROOT, no relativas al cwd del kernel)
os.makedirs(REPO_ROOT / 'data' / 'raw', exist_ok=True)
os.makedirs(REPO_ROOT / 'data' / 'interim', exist_ok=True)
os.makedirs(REPO_ROOT / 'data' / 'processed', exist_ok=True)

print(f"REPO_ROOT = {REPO_ROOT}")
print("Directorios configurados")

In [ ]:
import subprocess


def run_step(script_relpath, args=None, timeout=300):
    """Ejecuta un script de src/ con cwd=REPO_ROOT (para que sus rutas
    relativas internas, ej. 'data/raw/...', resuelvan correctamente) y
    RAISE si falla, en vez de solo imprimir una advertencia y seguir a la
    siguiente celda con datos que nunca se generaron."""
    cmd = [sys.executable, script_relpath] + (args or [])
    print(f"Ejecutando: {' '.join(cmd)}  (cwd={REPO_ROOT})")
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, cwd=str(REPO_ROOT))
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"{script_relpath} fallo con codigo {result.returncode}. Ver stderr arriba.")
    return result

## 2. GENERAR DATOS BASE (Simulación)

Ejecutar `simulation.py` para generar `movements_raw.csv` con transacciones realistas.

In [ ]:
print("Ejecutando simulacion de movimientos...")
run_step('src/simulation.py')

## 3. OBTENER CATÁLOGO NUTRICIONAL (Nueva: OpenFoodFacts)

Ejecutar `ingestion.py` para obtener datos nutricionales desde OpenFoodFacts API.

In [ ]:
print("Obteniendo catalogo desde OpenFoodFacts API...")
run_step('src/ingestion.py', timeout=300)

## 4. INYECTAR ANOMALÍAS REALISTAS

Crear versión "sucia" del dataset para simular datos del mundo real con problemas de calidad.

In [ ]:
print("Inyectando anomalias realistas en el dataset...")
run_step('src/anomaly_injection.py', [
    '--input', 'data/raw/movements_raw.csv',
    '--output', 'data/interim/movements_with_anomalies.csv',
    '--seed', '42',
    '--null_ratio', '0.15'
])

## 5. VERIFICAR ANOMALÍAS INYECTADAS

In [ ]:
import pandas as pd
import json

# Cargar bitácora de inyección
with open(REPO_ROOT / 'data' / 'interim' / 'movements_with_anomalies_injection_log.json', 'r') as f:
    injection_log = json.load(f)

print("ANOMALÍAS INYECTADAS:")
print(json.dumps(injection_log['anomalies'], indent=2))

# Cargar dataset con anomalías
df_anomalous = pd.read_csv(REPO_ROOT / 'data' / 'interim' / 'movements_with_anomalies.csv')
print(f"\nDataset con anomalías: {len(df_anomalous)} registros")
print(f"\nNulos por columna:")
print(df_anomalous.isna().sum())

## 6. LIMPIAR Y CORREGIR ANOMALÍAS

Ejecutar `preprocessing.py` para detectar y corregir todas las anomalías.
Genera bitácora de transformaciones y validación QA.

In [ ]:
print("Ejecutando pipeline de limpieza y corrección...")
run_step('src/preprocessing.py', [
    '--input', 'data/interim/movements_with_anomalies.csv',
    '--catalog', 'data/raw/catalog_raw.csv',
    '--output', 'data/processed/inventory_v1.csv',
    '--log', 'data/processed/inventory_v1_cleaning_log.json',
])

## 7. ANÁLISIS DE BITÁCORA DE LIMPIEZA

In [ ]:
# Cargar bitácora de limpieza
with open(REPO_ROOT / 'data' / 'processed' / 'inventory_v1_cleaning_log.json', 'r') as f:
    cleaning_log = json.load(f)

print("BITÁCORA DE LIMPIEZA Y CORRECCIÓN:")
print(json.dumps(cleaning_log, indent=2)[:2000] + "...")

print("\nResumen:")
print(f"Registros iniciales: {cleaning_log['registros_iniciales']['movements']}")
print(f"Registros finales: {cleaning_log['registros_finales']['movements']}")
print(f"Registros removidos: {cleaning_log['registros_iniciales']['movements'] - cleaning_log['registros_finales']['movements']}")

## 8. VALIDACIÓN DE DATOS LIMPIOS

In [ ]:
# Cargar dataset limpio
df_clean = pd.read_csv(REPO_ROOT / 'data' / 'processed' / 'inventory_v1.csv')

print(f"Dataset limpio cargado: {len(df_clean)} registros")
print(f"\nDimensiones:")
print(f"  Filas: {len(df_clean)}")
print(f"  Columnas: {len(df_clean.columns)}")

print(f"\nNulos después de limpieza:")
null_counts = df_clean.isna().sum()
if null_counts.sum() == 0:
    print("  No hay valores nulos")
else:
    print(null_counts[null_counts > 0])

print(f"\nPrimeras 5 filas:")
print(df_clean.head())

print(f"\nResumen estadístico:")
print(df_clean.describe())

## 9. COMPARACIÓN ANTES vs DESPUÉS

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Comparación: Dataset Sucio vs Limpio', fontsize=16, fontweight='bold')

# 1. Cantidad de registros
ax = axes[0, 0]
registros = [len(df_anomalous), len(df_clean)]
labels = ['Con Anomalías', 'Limpio']
ax.bar(labels, registros, color=['#ff6b6b', '#51cf66'])
ax.set_ylabel('Cantidad de registros')
ax.set_title('Registros antes y después')
for i, v in enumerate(registros):
    ax.text(i, v + 100, str(v), ha='center', fontweight='bold')

# 2. Nulos
ax = axes[0, 1]
nulls_before = df_anomalous.isna().sum().sum()
nulls_after = df_clean.isna().sum().sum()
nulls_data = [nulls_before, nulls_after]
ax.bar(labels, nulls_data, color=['#ff6b6b', '#51cf66'])
ax.set_ylabel('Cantidad de valores nulos')
ax.set_title('Valores nulos antes y después')
for i, v in enumerate(nulls_data):
    ax.text(i, v + 100, str(int(v)), ha='center', fontweight='bold')

# 3. % de nulos por columna (antes)
ax = axes[1, 0]
null_pct_before = (df_anomalous.isna().sum() / len(df_anomalous) * 100).sort_values(ascending=False)[:5]
ax.barh(range(len(null_pct_before)), null_pct_before.values, color='#ff6b6b')
ax.set_yticks(range(len(null_pct_before)))
ax.set_yticklabels(null_pct_before.index)
ax.set_xlabel('% de nulos')
ax.set_title('Top 5 columnas con más nulos (ANTES)')

# 4. % de nulos por columna (después)
ax = axes[1, 1]
null_pct_after = (df_clean.isna().sum() / len(df_clean) * 100).sort_values(ascending=False)[:5]
if len(null_pct_after) > 0 and null_pct_after.max() > 0:
    ax.barh(range(len(null_pct_after)), null_pct_after.values, color='#51cf66')
    ax.set_yticks(range(len(null_pct_after)))
    ax.set_yticklabels(null_pct_after.index)
ax.set_xlabel('% de nulos')
ax.set_title('Top 5 columnas con más nulos (DESPUÉS)')
ax.set_xlim(0, 30)

plt.tight_layout()
os.makedirs(REPO_ROOT / 'outputs', exist_ok=True)
plt.savefig(REPO_ROOT / 'outputs' / 'data_cleaning_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Gráfica guardada en: outputs/data_cleaning_comparison.png")

## 10. GENERACIÓN DE REPORTE DE CALIDAD DE DATOS

In [ ]:
# Crear reporte de QA
# NOTA: las claves de cleaning_log usadas abajo (registros_iniciales/finales,
# transformaciones.P6_deduplicacion, transformaciones.P3_integridad_referencial)
# son las que realmente produce src/preprocessing.py (ver la función
# run_preprocessing) — no 'initial_record_count'/'anomalies_detected', que
# nunca existieron en ese esquema y habrían provocado un KeyError aquí.
qa_report = {
    "ejecutado_el": pd.Timestamp.now().isoformat(),
    "resumen_general": {
        "dataset_limpio": "data/processed/inventory_v1.csv",
        "registros_totales": len(df_clean),
        "columnas": len(df_clean.columns),
        "porcentaje_completitud": float((1 - (df_clean.isna().sum().sum() / (len(df_clean) * len(df_clean.columns)))) * 100)
    },
    "anomalias_removidas": {
        "registros_removidos": cleaning_log['registros_iniciales']['movements'] - cleaning_log['registros_finales']['movements'],
        "duplicados": int(cleaning_log['transformaciones']['P6_deduplicacion']['duplicados_removidos']),
        "huerfanos": int(cleaning_log['transformaciones']['P3_integridad_referencial']['huerfanos_removidos'])
    },
    "integridad_datos": {
        "cantidad_nulos_totales": int(df_clean.isna().sum().sum()),
        "columnas_sin_nulos": int((df_clean.isna().sum() == 0).sum()),
        "datetime_range": {
            "inicio": str(df_clean['timestamp'].min()) if 'timestamp' in df_clean.columns else None,
            "fin": str(df_clean['timestamp'].max()) if 'timestamp' in df_clean.columns else None
        }
    },
    "validacion_tipos": {
        "all_quantity_numeric": bool(pd.api.types.is_numeric_dtype(df_clean['quantity'])),
        "all_timestamps_valid": bool(pd.api.types.is_datetime64_any_dtype(pd.to_datetime(df_clean['timestamp'], errors='coerce')))
    }
}

# Guardar reporte
os.makedirs(REPO_ROOT / 'docs', exist_ok=True)
with open(REPO_ROOT / 'docs' / 'data_quality_report.json', 'w') as f:
    json.dump(qa_report, f, indent=2, default=str)

print("Reporte de QA guardado en: docs/data_quality_report.json")
print("\nRESUMEN DE CALIDAD:")
print(json.dumps(qa_report, indent=2, default=str))

## ✅ PIPELINE COMPLETADO

### Archivos Generados

```
data/raw/
  ├── movements_raw.csv                    (original de simulación)
  ├── catalog_raw.csv                      (OpenFoodFacts)
  └── instacart_patterns.json              (patrones de referencia)

data/interim/
  ├── movements_with_anomalies.csv         (con anomalías inyectadas)
  └── movements_with_anomalies_injection_log.json

data/processed/
  ├── inventory_v1.csv                     (LIMPIO Y FINAL)
  └── inventory_v1_cleaning_log.json       (bitácora de transformaciones)

docs/
  └── data_quality_report.json             (reporte QA)

outputs/
  └── data_cleaning_comparison.png         (visualización antes/después)
```

### Próximos Pasos
1. ✅ **Ingestion.py** - Ahora obtiene datos de OpenFoodFacts
2. ✅ **Anomaly Injection** - Inyecta problemas realistas
3. ✅ **Preprocessing** - Detecta y limpia con bitácora
4. 📊 **Análisis Exploratorio** - Ver `01_perfilado.ipynb`
5. 📈 **Dashboard Tableau** - Usar `inventory_v1.csv`
